In [ ]:
                                                         # MCP

In [ ]:
# MCP is an intelligent middleware that connects LLMs with tools/APIs while managing context, structure, and execution.

In [3]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”  (Not using LLM)
# Scenario Background
# You are working in a company called ABC Corp.
# Employees face issues like:
# VPN not working
# Printer not responding
# Software errors
# Instead of calling IT support, employees use an AI Helpdesk Bot.
# What this Bot Should Do
# When a user types a problem:
# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee


# STEP 0: DATABASE (Simulated storage)
tickets_db = []  # This stores all tickets


# STEP 1: TOOL (MCP TOOL)
def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket
    

# STEP 2: AGENT REASONING (LLM SIMULATION)
def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority
    

# STEP 3: DECISION ENGINE (MCP CORE)
def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)

# STEP 4: MCP ORCHESTRATOR

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\nAgent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
         Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("Decision: No tool needed (AI response)")

        return "AI Response: Please describe your issue clearly."
        

# STEP 5: RUN INTERACTIVE LOOP
print("MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)


MCP Demo Started (Type 'exit' to stop)



Enter your query:  vpn issue



Agent received input: vpn issue
Decision: Tool call required
Extracted → Category: network, Priority: medium
MCP Payload: {'issue': 'vpn issue', 'priority': 'medium', 'category': 'network'}
Tool executed successfully

         Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: vpn issue
        Category: network
        Priority: medium
        


Enter your query:  software problem



Agent received input: software problem
Decision: Tool call required
Extracted → Category: general, Priority: medium
MCP Payload: {'issue': 'software problem', 'priority': 'medium', 'category': 'general'}
Tool executed successfully

         Ticket Created Successfully!

        Ticket ID: INC1001
        Issue: software problem
        Category: general
        Priority: medium
        


Enter your query:  exit


Exiting MCP demo...


In [7]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”  (using LLM)
# Scenario Background
# You are working in a company called ABC Corp.
# Employees face issues like:
# VPN not working
# Printer not responding
# Software errors
# Instead of calling IT support, employees use an AI Helpdesk Bot.
# What this Bot Should Do
# When a user types a problem:
# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee

import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Load API key securely
api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=api_key)

# STEP 0: DATABASE
tickets_db = []


# STEP 1: TOOL
def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# STEP 2: LLM ANALYSIS (REPLACES RULES)
def analyze_with_llm(user_input):
    """
    LLM decides:
    - should_create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # fast + powerful
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# STEP 3: MCP AGENT
def mcp_agent(user_input):

    print("\nAgent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
 Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return "AI Response: No ticket required. Try basic troubleshooting."


# STEP 4: RUN LOOP
print("LLM MCP Helpdesk Started (type 'exit')\n")

while True:

    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print("Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

LLM MCP Helpdesk Started (type 'exit')



Enter issue:  vpn issue



Agent received: vpn issue
LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
MCP Payload: {'issue': 'vpn issue', 'priority': 'medium', 'category': 'general'}

 Ticket Created Successfully!

Ticket ID: INC1000
Issue: vpn issue
Category: general
Priority: medium



Enter issue:  software problem



Agent received: software problem
LLM Decision: {'create_ticket': True, 'category': 'software', 'priority': 'medium'}
MCP Payload: {'issue': 'software problem', 'priority': 'medium', 'category': 'software'}

 Ticket Created Successfully!

Ticket ID: INC1001
Issue: software problem
Category: software
Priority: medium



...


Agent received: network problem
LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
MCP Payload: {'issue': 'network problem', 'priority': 'medium', 'category': 'general'}

 Ticket Created Successfully!

Ticket ID: INC1002
Issue: network problem
Category: general
Priority: medium



Enter issue:  General issue



Agent received: General issue
LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'low'}
MCP Payload: {'issue': 'General issue', 'priority': 'low', 'category': 'general'}

 Ticket Created Successfully!

Ticket ID: INC1003
Issue: General issue
Category: general
Priority: low



Enter issue:  exit


Exiting...


In [9]:
# MCP SCENARIO: “Smart HR Onboarding Assistant” (Not using LLM)
#  Scenario Background
# You are working in a company called XYZ Corp.
# New employees often face challenges during onboarding, such as:
# - Trouble accessing payroll portal
# - Confusion about leave policies
# - Difficulty setting up email accounts
# - Questions about training schedules
#  Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

#  What this Bot Should Do
# When a new hire types a question/problem:
# - Understand the query (e.g., “I can’t log into payroll”)
# - Decide if escalation to HR is needed
# - Identify:
# - Category (Payroll / Policy / IT Setup / Training)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, step-by-step instructions)
# - Show confirmation and next steps
# This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
# Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?
# STEP 0: DATABASE
tickets_db = []


# STEP 1: TOOL (MCP TOOL)
def create_ticket(issue, category, priority):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority
    }

    tickets_db.append(ticket)
    return ticket


# STEP 2: RULE-BASED ANALYSIS (NO LLM)
def analyze_input(user_input):

    text = user_input.lower()

    # 🔹 Category Detection
    if "payroll" in text:
        category = "payroll"
    elif "leave" in text or "policy" in text:
        category = "policy"
    elif "email" in text or "login" in text:
        category = "it_setup"
    elif "training" in text:
        category = "training"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text or "later" in text:
        priority = "low"
    else:
        priority = "medium"

    # Decision (create ticket or not)
    keywords = ["not working", "can't", "cannot", "issue", "problem"]

    create_ticket_flag = any(word in text for word in keywords)

    return {
        "create_ticket": create_ticket_flag,
        "category": category,
        "priority": priority
    }


# STEP 3: MCP AGENT (ORCHESTRATOR)
def mcp_agent(user_input):

    print("\nProcessing:", user_input)

    # Step 3.1: Analyze input
    decision = analyze_input(user_input)

    print("Decision:", decision)

    # Step 3.2: If ticket needed
    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "category": decision["category"],
            "priority": decision["priority"]
        }

        print("MCP Payload:", payload)

        # Step 3.3: Tool call
        result = create_ticket(**payload)

        # Step 3.4: Response
        return f"""
Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

HR will contact you soon.
"""

    else:
        # Step 3.5: Direct response (no tool)
        return f"""
        AI Guidance:

for {decision['category']} queries, please check HR portal or FAQs.
"""


# STEP 4: RUN LOOP
print("HR Assistant (No API) Started (type 'exit')\n")

while True:

    user_input = input("Ask HR Bot: ")

    if user_input.lower() == "exit":
        print("Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

HR Assistant (No API) Started (type 'exit')



Ask HR Bot:  Payroll login not working urgently



Processing: Payroll login not working urgently
Decision: {'create_ticket': True, 'category': 'payroll', 'priority': 'high'}
MCP Payload: {'issue': 'Payroll login not working urgently', 'category': 'payroll', 'priority': 'high'}

Ticket Created!

Ticket ID: HR1000
Category: payroll
Priority: high

HR will contact you soon.



Ask HR Bot:  exit


Exiting...


In [8]:
# MCP SCENARIO: “Smart HR Onboarding Assistant”  (Using LLM)
#  Scenario Background
# You are working in a company called XYZ Corp.
# New employees often face challenges during onboarding, such as:
# - Trouble accessing payroll portal
# - Confusion about leave policies
# - Difficulty setting up email accounts
# - Questions about training schedules
#  Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

#  What this Bot Should Do
# When a new hire types a question/problem:
# - Understand the query (e.g., “I can’t log into payroll”)
# - Decide if escalation to HR is needed
# - Identify:
# - Category (Payroll / Policy / IT Setup / Training)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, step-by-step instructions)
# - Show confirmation and next steps
# This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
# Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("API key not found. Check your .env file")

client = Groq(api_key=api_key)

# DATABASE
tickets_db = []

# TOOL (MCP TOOL)
def create_ticket(issue, category, priority):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority
    }

    tickets_db.append(ticket)
    return ticket


# LLM ANALYSIS (MCP BRAIN)
def analyze_with_llm(user_input):

    prompt = f"""
You are an HR onboarding assistant.

Analyze the user query and respond ONLY in JSON format:

{{
  "create_ticket": true/false,
  "category": "payroll/policy/it_setup/training/general",
  "priority": "high/medium/low",
  "response": "short helpful message for user"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium",
            "response": "We are looking into your issue."
        }

    return parsed


# MCP AGENT
def mcp_agent(user_input):

    print("\nProcessing:", user_input)

    decision = analyze_with_llm(user_input)

    print("LLM Decision:", decision)

    # If ticket needed
    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "category": decision["category"],
            "priority": decision["priority"]
        }

        print("MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

Message: {decision['response']}
Next Step: HR team will contact you soon.
"""

    else:
        return f"""
 AI Guidance:

{decision['response']}
"""

# RUN LOOP
print("Smart HR Onboarding Assistant Started (type 'exit')\n")

while True:

    user_input = input("Ask HR Bot: ")

    if user_input.lower() == "exit":
        print("Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

Smart HR Onboarding Assistant Started (type 'exit')



Ask HR Bot:  Payroll login not working urgently



Processing: Payroll login not working urgently
LLM Decision: {'create_ticket': True, 'category': 'payroll', 'priority': 'high', 'response': "Sorry to hear that your payroll login is not working. We'll assist you urgently to resolve the issue. Please try resetting your password or contact IT support for immediate help."}
MCP Payload: {'issue': 'Payroll login not working urgently', 'category': 'payroll', 'priority': 'high'}

Ticket Created!

Ticket ID: HR1000
Category: payroll
Priority: high

Message: Sorry to hear that your payroll login is not working. We'll assist you urgently to resolve the issue. Please try resetting your password or contact IT support for immediate help.
Next Step: HR team will contact you soon.



Ask HR Bot:  How many holidays do we get?



Processing: How many holidays do we get?
LLM Decision: {'create_ticket': False, 'category': 'policy', 'priority': 'low', 'response': 'You can find the holiday calendar and policy details on our company intranet or reach out to your supervisor for more information.'}

 AI Guidance:

You can find the holiday calendar and policy details on our company intranet or reach out to your supervisor for more information.



Ask HR Bot:  exit


Exiting...


In [12]:
# MCP SCENARIO: “Smart Banking Support Assistant” (not Using LLM)
#  Scenario Background
# You are working in a company called FinTrust Bank.
# Customers often face issues such as:
# - Credit card not working
# - Trouble with online banking login
# - Queries about loan status
# - Transaction disputes
#  Instead of calling customer care, customers use an AI Banking Support Bot.

#  What this Bot Should Do
# When a customer types a problem:
# - Understand the issue (e.g., “My card was declined”)
# - Decide if escalation to a human agent is needed
# - Identify:
# - Category (Card Services / Online Banking / Loans / Transactions)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, troubleshooting steps, policy info)
# - Show confirmation and next steps
# This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
# Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

# STEP 0: DATABASE (Simulated storage)
tickets_db = []


# STEP 1: TOOL (MCP TOOL)
def create_ticket(issue, priority, category):
    """
    Simulates banking support ticket system
    """

    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# STEP 2: AGENT REASONING (RULE-BASED)
def analyze_input(user_input):
    """
    Extract category and priority using rules
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "card" in text:
        category = "card services"
    elif "login" in text or "banking" in text:
        category = "online banking"
    elif "loan" in text:
        category = "loans"
    elif "transaction" in text or "money" in text:
        category = "transactions"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "failed" in text or "not working" in text:
        priority = "medium"
    else:
        priority = "low"

    return category, priority


# STEP 3: DECISION ENGINE (MCP CORE)
def should_call_tool(user_input):
    """
    Decide if ticket is needed
    """

    keywords = ["not working", "failed", "issue", "problem", "declined"]

    return any(word in user_input.lower() for word in keywords)


# STEP 4: MCP ORCHESTRATOR
def mcp_agent(user_input):
    """
    Main MCP flow
    """

    print("\nAgent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: MCP Payload
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        print("Decision: No tool needed")

        return "AI Response: Please check FAQs or provide more details."


# STEP 5: RUN LOOP
print("Banking MCP Assistant Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

Banking MCP Assistant Started (Type 'exit' to stop)



Enter your query:  My credit card is not working



Agent received input: My credit card is not working
Decision: Tool call required
Extracted → Category: card services, Priority: medium
MCP Payload: {'issue': 'My credit card is not working', 'priority': 'medium', 'category': 'card services'}
Tool executed successfully

Ticket Created Successfully!

Ticket ID: BNK1000
Issue: My credit card is not working
Category: card services
Priority: medium



Enter your query:  exit


Exiting...


In [10]:
# MCP SCENARIO: “Smart Banking Support Assistant” (Using LLM)
#  Scenario Background
# You are working in a company called FinTrust Bank.
# Customers often face issues such as:
# - Credit card not working
# - Trouble with online banking login
# - Queries about loan status
# - Transaction disputes
#  Instead of calling customer care, customers use an AI Banking Support Bot.

#  What this Bot Should Do
# When a customer types a problem:
# - Understand the issue (e.g., “My card was declined”)
# - Decide if escalation to a human agent is needed
# - Identify:
# - Category (Card Services / Online Banking / Loans / Transactions)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, troubleshooting steps, policy info)
# - Show confirmation and next steps
# This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
# Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("API key not found. Check your .env file")

client = Groq(api_key=api_key)

# DATABASE
tickets_db = []

# TOOL (MCP TOOL)
def create_ticket(issue, category, priority):
    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority
    }

    tickets_db.append(ticket)
    return ticket


# LLM ANALYSIS (MCP BRAIN)
def analyze_with_llm(user_input):

    prompt = f"""
You are a banking support assistant.

Analyze the user query and respond ONLY in JSON format:

{{
  "create_ticket": true/false,
  "category": "card_services/online_banking/loans/transactions/general",
  "priority": "high/medium/low",
  "response": "short helpful message for customer"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium",
            "response": "We are checking your issue."
        }

    return parsed


# MCP AGENT
def mcp_agent(user_input):

    print("\nProcessing:", user_input)

    decision = analyze_with_llm(user_input)

    print("LLM Decision:", decision)

    # If ticket needed
    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "category": decision["category"],
            "priority": decision["priority"]
        }

        print("MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

Message: {decision['response']}
Next Step: Our support team will contact you soon.
"""

    else:
        return f"""
AI Guidance:

{decision['response']}
"""


# RUN LOOP
print("Smart Banking Support Assistant Started (type 'exit')\n")

while True:

    user_input = input("Ask Banking Bot: ")

    if user_input.lower() == "exit":
        print("Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

Smart Banking Support Assistant Started (type 'exit')



Ask Banking Bot:  My credit card is not working



Processing: My credit card is not working
LLM Decision: {'create_ticket': True, 'category': 'card_services', 'priority': 'high', 'response': "Sorry to hear that your credit card is not working. We'll look into this issue and get back to you soon. Please check if your card is expired or if you have sufficient funds."}
MCP Payload: {'issue': 'My credit card is not working', 'category': 'card_services', 'priority': 'high'}

Ticket Created!

Ticket ID: BNK1000
Category: card_services
Priority: high

Message: Sorry to hear that your credit card is not working. We'll look into this issue and get back to you soon. Please check if your card is expired or if you have sufficient funds.
Next Step: Our support team will contact you soon.



Ask Banking Bot:  I can’t login to banking



Processing: I can’t login to banking
LLM Decision: {'create_ticket': True, 'category': 'online_banking', 'priority': 'high', 'response': "Sorry to hear that you're having trouble logging in. Please try resetting your password or clearing your browser cache. If issues persist, we'll be happy to assist you further."}
MCP Payload: {'issue': 'I can’t login to banking', 'category': 'online_banking', 'priority': 'high'}

Ticket Created!

Ticket ID: BNK1001
Category: online_banking
Priority: high

Message: Sorry to hear that you're having trouble logging in. Please try resetting your password or clearing your browser cache. If issues persist, we'll be happy to assist you further.
Next Step: Our support team will contact you soon.



Ask Banking Bot:  exit


Exiting...


In [14]:
# Create a  Weather Tool MCP Server that any AI agent can use with sample use case

# MCP WEATHER TOOL WITH LLM
import json
import random

# STEP 1: TOOL (Weather API Simulation)
def get_weather(city):
    conditions = ["Sunny", "Rainy", "Cloudy", "Stormy"]

    return {
        "city": city,
        "temperature": random.randint(18, 40),
        "condition": random.choice(conditions),
        "humidity": random.randint(40, 90)
    }


# STEP 2: LLM SIMULATION (IMPORTANT)
def call_llm(user_input):
    """
    Simulates LLM response (like OpenAI / GPT)
    Returns JSON decision
    """

    text = user_input.lower()

    if "weather" in text or "temperature" in text or "rain" in text:
        city = user_input.split()[-1]

        return json.dumps({
            "action": "get_weather",
            "parameters": {
                "city": city
            }
        })

    else:
        return json.dumps({
            "action": "none",
            "response": "Please ask weather-related questions."
        })


# STEP 3: MCP AGENT
def mcp_agent(user_input):

    print("\nUser Input:", user_input)

    # STEP 3.1  Send to LLM
    llm_output = call_llm(user_input)
    print("LLM Output:", llm_output)

    # STEP 3.2  Parse JSON
    decision = json.loads(llm_output)

    # STEP 3.3  MCP Decision
    if decision["action"] == "get_weather":

        print("MCP Decision: Call Weather Tool")

        params = decision["parameters"]
        print("Parameters:", params)

        # STEP 3.4 → Tool Call
        result = get_weather(**params)
        print("Tool executed")

        # STEP 3.5 → Final Response
        return f"""
Weather Report 

City: {result['city']}
Temperature: {result['temperature']}degree C
Condition: {result['condition']}
Humidity: {result['humidity']}%
"""

    else:
        print("MCP Decision: No tool call")
        return decision["response"]


# STEP 4: RUN PROGRAM
print("MCP + LLM Weather Assistant Started (type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("Stopping...")
        break

    response = mcp_agent(user_input)
    print(response)

MCP + LLM Weather Assistant Started (type 'exit' to stop)



Enter your query:  Can you tell me the weather in Hyderabad



User Input: Can you tell me the weather in Hyderabad
LLM Output: {"action": "get_weather", "parameters": {"city": "Hyderabad"}}
MCP Decision: Call Weather Tool
Parameters: {'city': 'Hyderabad'}
Tool executed

Weather Report 

City: Hyderabad
Temperature: 18degree C
Condition: Cloudy
Humidity: 79%



Enter your query:  exit


Stopping...


In [24]:
# MCP + LLM WEATHER APP (FINAL FINAL FIX)


import requests
import json
import gradio as gr
import os
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

API_KEY = os.getenv("mykey") 

# print("Loaded API KEY:", API_KEY)

def get_weather(city):

    if not API_KEY:
        return "API key not found!"

  
    url = f"https://api.openweathermap.org/data/2.5/forecast?q={city},IN&appid={API_KEY}&units=metric"

    try:
        response = requests.get(url)
        data = response.json()

        print("API Response:", data)  # DEBUG

        if str(data.get("cod")) != "200":
            return f"Error: {data.get('message')}"

        forecast = {}

        for item in data["list"]:
            date = item["dt_txt"].split(" ")[0]
            day_name = datetime.strptime(date, "%Y-%m-%d").strftime("%A")

            if day_name not in forecast:
                forecast[day_name] = {
                    "temp": item["main"]["temp"],
                    "condition": item["weather"][0]["description"]
                }

        result = ""
        for day in ["Monday", "Tuesday"]:
            if day in forecast:
                result += f"{day} → {forecast[day]['temp']}°C, {forecast[day]['condition']}\n"

        return result if result else "No data for Monday/Tuesday"

    except Exception as e:
        return f"Error occurred: {str(e)}"

def call_llm(user_input):
    text = user_input.lower()

    if any(word in text for word in ["weather", "temperature", "rain"]):

        
        if "in" in text:
            city = user_input.lower().split("in", 1)[1].strip()
        else:
            city = user_input.strip()

        return json.dumps({
            "action": "get_weather",
            "parameters": {"city": city}
        })

    else:
        return json.dumps({
            "action": "none",
            "response": "Ask something like: weather in Delhi"
        })

def mcp_agent(user_input):

    llm_output = call_llm(user_input)
    decision = json.loads(llm_output)

    if decision["action"] == "get_weather":
        result = get_weather(**decision["parameters"])
        return f"🌦️ Weather Forecast:\n\n{result}"
    else:
        return decision["response"]

interface = gr.Interface(
    fn=mcp_agent,
    inputs=gr.Textbox(placeholder="Ask: weather in Delhi"),
    outputs="text",
    title="🌦️ MCP Weather Assistant",
    description="LLM + MCP + OpenWeather API"
)

interface.launch()

Loaded API KEY: ca385d85e3f44a1d1ec03f6eaff89ba2
* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
